In [13]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv(
    "uber_weather_enriched.csv",
    parse_dates=['pickup_datetime']
)

df.head()

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,temperature,precipitation,wind_speed
0,2009-01-01 01:15:22.0000006,8.5,2009-01-01 01:15:22+00:00,-73.981918,40.779456,-73.957685,40.771043,2,2009-01-01 01:00:00,-7.2,0.0,34.1
1,2009-01-01 01:59:17.0000001,13.0,2009-01-01 01:59:17+00:00,-73.983759,40.721389,-73.994833,40.687179,2,2009-01-01 01:00:00,-7.2,0.0,34.1
2,2009-01-01 02:05:03.0000003,10.6,2009-01-01 02:05:03+00:00,-73.956635,40.771254,-73.991528,40.749778,2,2009-01-01 02:00:00,-7.3,0.0,33.6
3,2009-01-01 02:14:20.0000003,5.0,2009-01-01 02:14:20+00:00,-73.986486,40.734734,-73.983508,40.730138,1,2009-01-01 02:00:00,-7.3,0.0,33.6
4,2009-01-01 02:09:13.0000003,12.2,2009-01-01 02:09:13+00:00,-73.984605,40.728020,-73.955746,40.776830,1,2009-01-01 02:00:00,-7.3,0.0,33.6


In [19]:
required_cols = [
    'pickup_latitude',
    'pickup_longitude',
    'dropoff_latitude',
    'dropoff_longitude',
    'pickup_datetime'
]

print(df[required_cols].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   pickup_latitude    200000 non-null  float64            
 1   pickup_longitude   200000 non-null  float64            
 2   dropoff_latitude   199999 non-null  float64            
 3   dropoff_longitude  199999 non-null  float64            
 4   pickup_datetime    200000 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(4)
memory usage: 7.6 MB
None


In [20]:
coords = [
    'pickup_latitude',
    'pickup_longitude',
    'dropoff_latitude',
    'dropoff_longitude'
]

# Convert datatype
for col in coords:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove null values
df.dropna(subset=coords, inplace=True)

# Remove invalid coordinates
df = df[
    df['pickup_latitude'].between(-90, 90) &
    df['dropoff_latitude'].between(-90, 90) &
    df['pickup_longitude'].between(-180, 180) &
    df['dropoff_longitude'].between(-180, 180)
]

df.shape

(199987, 12)

In [21]:
def haversine(lat1, lon1, lat2, lon2):

    R = 6371  # Earth radius (km)

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat/2)**2 +
        np.cos(lat1)*np.cos(lat2)*
        np.sin(dlon/2)**2
    )

    # numerical stability
    a = np.clip(a, 0, 1)

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [22]:
df['distance_km'] = haversine(
    df['pickup_latitude'].values,
    df['pickup_longitude'].values,
    df['dropoff_latitude'].values,
    df['dropoff_longitude'].values
)

df[['distance_km']].head()

,distance_km
0,2.244765
1,3.916842
2,3.786736
3,0.569331
4,5.946957


In [23]:
df = df[
    (df['distance_km'] > 0) &
    (df['distance_km'] < 100)
]

df.shape

(193895, 13)

In [24]:
df['hour'] = df['pickup_datetime'].dt.hour

In [25]:
df['is_peak'] = df['hour'].isin(
    [7,8,9,16,17,18,19]
).astype(int)

df[['hour','is_peak']].head()

,hour,is_peak
0,1,0
1,1,0
2,2,0
3,2,0
4,2,0


In [26]:
df['duration_min'] = (
    df['distance_km'] / 30
) * 60

In [27]:
df['traffic_duration_min'] = np.where(
    df['is_peak'] == 1,
    df['duration_min'] * 1.6,   # heavy traffic
    df['duration_min'] * 1.2    # normal traffic
)

In [28]:
df['traffic_delay'] = (
    df['traffic_duration_min']
    - df['duration_min']
)

In [29]:
df['avg_speed'] = (
    df['distance_km'] /
    (df['traffic_duration_min'] / 60)
)

In [30]:
df[[
    'distance_km',
    'duration_min',
    'traffic_duration_min',
    'traffic_delay',
    'avg_speed'
]].describe()

,distance_km,duration_min,traffic_duration_min,traffic_delay,avg_speed
count,193895.000000,193895.000000,193895.000000,193895.000000,193895.000000
mean,3.361433,6.722867,8.930230,2.207363,22.863921
std,3.728248,7.456496,10.040252,3.045697,2.964406
min,0.000084,0.000168,0.000202,0.000034,18.750000
25%,1.279831,2.559661,3.389333,0.669854,18.750000
50%,2.179475,4.358950,5.797243,1.291798,25.000000
75%,3.937333,7.874666,10.480684,2.553931,25.000000
max,99.162509,198.325019,313.824977,117.684367,25.000000


In [31]:
df.to_csv(
    "final_weather_traffic_dataset.csv",
    index=False
)

print("✅ Traffic features successfully added!")

✅ Traffic features successfully added!
